# CucuGAN: Struktur-Loss-Kurztest

Vergleichslauf auf Kaggle: 2.000 Bilder pro Domäne, 20 Epochen, `lambda_struct=3`.
Der Lauf verwendet keine Decay-Epochen und überschreibt keinen älteren Lauf.

In [ ]:
from pathlib import Path
import os
import shutil

# Kaggle-Dataset mit der hochgeladenen ZIP durchsuchen und entpacken.
input_files = list(Path('/kaggle/input').rglob('*.zip'))
if input_files:
    archive = input_files[0]
    extracted = Path('/kaggle/working/data')
    shutil.unpack_archive(archive, extracted)
    candidates = list(extracted.rglob('squares128'))
    DATA_ROOT = candidates[0] if candidates else extracted
else:
    DATA_ROOT = Path('/kaggle/input/cucugan/squares128')
REPO_DIR = Path('/kaggle/working/CucuGAN')
RUN_NAME = 'struct_test_20'
print('Kaggle input:')
for path in sorted(Path('/kaggle/input').iterdir()):
    print(' ', path)
print('DATA_ROOT:', DATA_ROOT)


In [ ]:
import subprocess
if not (REPO_DIR / 'src' / 'train.py').exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Felix-uniulm/CucuGAN.git', str(REPO_DIR)
    ], check=True)
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-r',
    str(REPO_DIR / 'requirements.txt')
], check=True)assert (DATA_ROOT / 'train' / 'A').is_dir(), f'Fehlt: {DATA_ROOT}/train/A'
assert (DATA_ROOT / 'train' / 'B').is_dir(), f'Fehlt: {DATA_ROOT}/train/B'

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
os.chdir(REPO_DIR)
subprocess.run([
    'python', '-u', 'src/train.py',
    '--name', RUN_NAME,
    '--data', str(DATA_ROOT),
    '--max-images', '2000',
    '--n-epochs', '20',
    '--n-epochs-decay', '0',
    '--lambda-struct', '3',
    '--save-every', '1',
    '--batch-size', '1',
    '--pool-size', '50',
    '--device', 'cuda',
    '--num-workers', '2',
], check=True)

In [ ]:
from IPython.display import display
from PIL import Image

sample_dir = REPO_DIR / 'outputs' / RUN_NAME / 'samples'
for epoch in (1, 5, 10, 15, 20):
    path = sample_dir / f'epoch_{epoch:03d}.png'
    if path.exists():
        print(path)
        display(Image.open(path))